# 🚔 Análise de Apreensões — Pandas, NumPy, Matplotlib e Seaborn

## 🎯 Objetivos deste Desafio

Ao final deste notebook, você será capaz de:
- ✅ Carregar e explorar uma base de ocorrências em Excel (Google Colab)
- ✅ Aplicar **anonimização** de CPF e nome com funções personalizadas
- ✅ Realizar **tratamento lógico** de dados inválidos (sem apagar linhas)
- ✅ Identificar o **registro representativo único** de cada apreensão (`REP_DADO_UNICO`)
- ✅ Produzir insights com **gráficos e tabelas** (Matplotlib + Seaborn + Pandas)

---

## 📚 Pré-requisitos
- Fundamentos de **NumPy** e **Pandas** (Notebook 1.1)
- Fundamentos de **Matplotlib** e **Seaborn** (Notebook 1.2)
- Arquivo Excel com as colunas da base de apreensões

---

## 📖 Agenda deste notebook

1. **🛠️ Preparação do Ambiente e Carregamento dos Dados**
2. **🔍 Exploração Inicial (EDA rápida)**
3. **🎭 Anonimização de CPF e Nome**
4. **🧹 Tratamento Lógico com `status_dado`**
5. **🧩 Registro Representativo Único (`REP_DADO_UNICO`)**
6. **📈 Análises e Visualizações**
   - ⏳ Distribuição temporal das principais apreensões
   - 🗺️ Distribuição espacial por UF
   - 🛣️ Rodovias mais relevantes
7. **🎓 Conclusão e Cheat Sheet**

---

> **💡 Contexto:** Combinações de `DS_GRUPO_ENQTO` × `DS_TIPO_APREENSAO` podem gerar **produto cartesiano** (linhas multiplicadas). Por isso, totais de apreensão devem usar apenas linhas com `REP_DADO_UNICO == 'U'`.


# 🛠️ 1. Preparação do Ambiente e Carregamento dos Dados

### 📦 Bibliotecas que vamos usar
- 🧮 **NumPy** — operações numéricas e estatísticas
- 📊 **Pandas** — leitura, limpeza, agrupamentos e tabelas
- 📈 **Matplotlib** — gráficos base (linhas, barras, etc.)
- 🎨 **Seaborn** — gráficos estatísticos mais elegantes

### 📁 Como carregar o Excel no Google Colab
1. Execute a célula de upload abaixo
2. Escolha o arquivo `.xlsx` / `.xls` no seu computador
3. O Pandas lerá a planilha automaticamente

> **⚠️ Dica:** Se estiver rodando **fora** do Colab, ajuste o caminho do arquivo na variável `CAMINHO_ARQUIVO`.


In [ ]:
# 📦 Importando as bibliotecas necessárias
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# 🎨 Configurações para gráficos mais limpos e legíveis
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid', palette='deep')

print('✅ Bibliotecas carregadas com sucesso!')
print(f'🧮 NumPy: {np.__version__}')
print(f'📊 Pandas: {pd.__version__}')
print(f'🎨 Seaborn: {sns.__version__}')


In [ ]:
# 📥 Carregamento do Excel
# =============================================================================
# ▶️ GOOGLE COLAB (recomendado para a entrega):
# 1) Descomente o bloco abaixo
# 2) Execute a célula e escolha o arquivo .xlsx no seletor
# =============================================================================
# from google.colab import files
# uploaded = files.upload()  # abre o seletor de arquivos
# CAMINHO_ARQUIVO = list(uploaded.keys())[0]

# =============================================================================
# ▶️ ALTERNATIVA: caminho manual (local ou /content no Colab)
# =============================================================================
CAMINHO_ARQUIVO = 'base_apreensoes.xlsx'  # 👈 nome do seu arquivo

# Fallback didático (se existir no mesmo diretório)
if not Path(CAMINHO_ARQUIVO).exists():
    candidato = Path('base_apreensoes_exemplo.xlsx')
    if candidato.exists():
        CAMINHO_ARQUIVO = str(candidato)
        print('ℹ️ Arquivo principal não encontrado. Usando base de exemplo para demonstração.')
    else:
        raise FileNotFoundError(
            f'Arquivo não encontrado: {CAMINHO_ARQUIVO}\n'
            'Faça upload no Colab (files.upload) ou ajuste CAMINHO_ARQUIVO.'
        )

# Leitura
df = pd.read_excel(CAMINHO_ARQUIVO)

print('📂 Arquivo carregado:', CAMINHO_ARQUIVO)
print(f'📏 Dimensões: {df.shape[0]} linhas × {df.shape[1]} colunas')
print('\n📋 Colunas disponíveis:')
for col in df.columns:
    print(f'  • {col}')


# 🔍 2. Exploração Inicial (EDA rápida)

Antes de transformar, **conheça** os dados:
- 👀 `head()` / `tail()` — amostra
- 🧾 `info()` — tipos e nulos
- 📊 `describe()` — estatísticas
- 🔎 `isnull().sum()` — faltantes

> **💡 Lembrete da aula:** limpeza boa começa com diagnóstico bom!


In [ ]:
# 👀 Amostra das primeiras linhas
print('🔍 Primeiras 5 linhas:')
df.head()


In [ ]:
# 🧾 Estrutura, tipos e memória
print('🧾 Informações do DataFrame:\n')
df.info()


In [ ]:
# 🔎 Dados ausentes por coluna
print('🧹 Valores ausentes (isnull) por coluna:')
ausentes = df.isnull().sum().sort_values(ascending=False)
display(ausentes[ausentes > 0] if ausentes.sum() > 0 else '✅ Nenhum nulo encontrado com isnull()')

# Estatísticas das colunas numéricas (quando existirem)
print('\n📊 Estatísticas descritivas (numéricas):')
df.describe(include=[np.number])


In [ ]:
# 🔧 Padronização leve dos nomes de colunas (espaços extras)
# Mantém os nomes originais o mais próximo possível do enunciado

df.columns = [str(c).strip() for c in df.columns]

# Garante presença das colunas essenciais (falha cedo se faltar alguma)
COLUNAS_ESSENCIAIS = [
    'NU_CPF_PESSOA',
    'DS_DATA_CURTA',
    'NO_ENVOLVIDO',
    'DS_GRUPO_ENQTO',
    'DS_RODOVIA',
    'DS_UF_LOCAL_OCORRENCIA',
    'SG_UF_LOCAL_OCORRENCIA',
    'QTDE APREENSâO OCORRÊNCIA',
    'DS_TIPO_APREENSAO',
]

faltantes = [c for c in COLUNAS_ESSENCIAIS if c not in df.columns]
if faltantes:
    print('⚠️ Atenção: colunas esperadas não encontradas:')
    for c in faltantes:
        print(f'  - {c}')
    print('\n📋 Colunas atuais:', list(df.columns))
    print('\n💡 Se o nome da quantidade vier com grafia diferente (ex.: APREENSÃO),')
    print('   ajuste o dicionário de renomeação abaixo.')
else:
    print('✅ Todas as colunas essenciais foram encontradas!')

# Mapeamento opcional para variações comuns de grafia
renomear = {}
for c in df.columns:
    c_norm = unicodedata.normalize('NFKD', c).encode('ascii', 'ignore').decode('ascii').upper()
    if 'QTDE' in c_norm and 'APREENS' in c_norm and 'OCORREN' in c_norm:
        renomear[c] = 'QTDE APREENSâO OCORRÊNCIA'
    if c_norm in {'UF_LOCAL_OCORRENCIA', 'DS_UF_LOCAL_OCORRENCIA'}:
        renomear[c] = 'DS_UF_LOCAL_OCORRENCIA'
    if c_norm in {'SG_UF_LOCAL_OCORRENCIA', 'SG_UF'}:
        renomear[c] = 'SG_UF_LOCAL_OCORRENCIA'

if renomear:
    df = df.rename(columns=renomear)
    print('🔁 Colunas padronizadas:', renomear)


# 🎭 3. Anonimização de CPF e Nome

## 🎯 Regras do desafio

| Campo | Regra | Exemplo |
|------|------|---------|
| **CPF** (`NU_CPF_PESSOA`) | Manter 3 primeiros dígitos + máscara + último dígito | `123.***.***-*5` |
| **Nome** (`NO_ENVOLVIDO`) | Iniciais em caixa alta (ignorando partículas) | `Joao Jose da Silva Nogueira` → `JJSN` |

Os resultados vão para **novas colunas** (os originais permanecem na base bruta para auditoria interna — mas nos gráficos usaremos só as versões anonimizadas quando fizer sentido).

> **🔐 Boas práticas:** em relatórios públicos, prefira sempre exibir `CPF_ANON` e `NOME_INICIAIS`.


In [ ]:
# 🎭 Funções de anonimização

PARTICULAS_NOME = {
    'DA', 'DE', 'DO', 'DAS', 'DOS', 'E', 'A', 'O', 'AS', 'OS',
    'DI', 'DU', 'DEL', 'DELLA', 'VAN', 'VON'
}


def apenas_digitos(valor):
    """Extrai somente dígitos de um valor (CPF)."""
    if pd.isna(valor):
        return ''
    return re.sub(r'\D', '', str(valor))


def anonimizar_cpf(cpf) -> object:
    """Anonimiza CPF no formato 123.***.***-*5.

    - Usa os 3 primeiros dígitos
    - Mascara o meio com ***.***-*
    - Mantém apenas o último dígito
    """
    digitos = apenas_digitos(cpf)
    if len(digitos) < 4:
        return None
    return f"{digitos[:3]}.***.***-*{digitos[-1]}"


def iniciais_nome(nome) -> object:
    """Gera iniciais em caixa alta, ignorando partículas (da, de, do...).

    Ex.: 'Joao Jose da Silva Nogueira' -> 'JJSN'
    """
    if pd.isna(nome):
        return None
    texto = str(nome).strip()
    if not texto:
        return None

    # Remove acentos para robustez, mas preserva letras
    tokens = re.findall(r"[A-Za-zÀ-ÿ]+", texto)
    iniciais = []
    for token in tokens:
        token_up = unicodedata.normalize('NFKD', token).encode('ascii', 'ignore').decode('ascii').upper()
        if token_up in PARTICULAS_NOME:
            continue
        if token_up:
            iniciais.append(token_up[0])
    return ''.join(iniciais) if iniciais else None


# ✅ Testes rápidos (como na aula — valide a função antes de aplicar em massa!)
print('🧪 Testes de anonimização:')
print(' CPF 123.456.789-05 ->', anonimizar_cpf('123.456.789-05'))
print(' CPF 12345678905    ->', anonimizar_cpf('12345678905'))
print(' Nome exemplo       ->', iniciais_nome('Joao Jose da Silva Nogueira'))
print(' Nome com acento    ->', iniciais_nome('José Antônio de Oliveira'))


In [ ]:
# 🆕 Criando colunas anonimizadas
df['CPF_ANON'] = df['NU_CPF_PESSOA'].apply(anonimizar_cpf)
df['NOME_INICIAIS'] = df['NO_ENVOLVIDO'].apply(iniciais_nome)

print('✅ Colunas criadas: CPF_ANON e NOME_INICIAIS')
df[['NU_CPF_PESSOA', 'CPF_ANON', 'NO_ENVOLVIDO', 'NOME_INICIAIS']].head(10)


# 🧹 4. Tratamento Lógico com `status_dado`

## 🎯 Ideia-chave
**Não vamos apagar linhas** agora. Em vez disso, criamos uma coluna de status que decide se o registro entra nos relatórios/gráficos.

### ❌ CPF considerado inválido para apresentação quando:
- estiver em branco (`''`)
- for nulo (`NaN` / `None`)
- contiver textos como **"nao informado"** / **"não informado"** / **"ni"** etc.

| `status_dado` | Significado |
|---|---|
| `APRESENTAR` | Linha apta para relatórios e gráficos |
| `NAO_APRESENTAR` | Linha mantida na base, mas fora das análises |

> **💡 Técnica da aula:** criar colunas derivadas com `.apply()` / condições vetorizadas é um clássico de limpeza com Pandas!


In [ ]:
# 🧹 Função de classificação lógica do CPF

INVALIDOS_TEXTO = {
    'nao informado', 'não informado', 'nao_informado', 'não_informado',
    'naoinformado', 'nãoinformado', 'ni', 'n/i', 'n.i.', 'n.i',
    'sem informacao', 'sem informação', 'null', 'none', 'nan', '-', '--', 'n/a'
}


def classificar_status_cpf(cpf):
    """Retorna APRESENTAR ou NAO_APRESENTAR com base no CPF."""
    if pd.isna(cpf):
        return 'NAO_APRESENTAR'

    texto = str(cpf).strip()
    if texto == '':
        return 'NAO_APRESENTAR'

    texto_norm = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii')
    texto_norm = texto_norm.strip().lower()
    if texto_norm in INVALIDOS_TEXTO:
        return 'NAO_APRESENTAR'

    digitos = apenas_digitos(texto)
    # CPF brasileiro tem 11 dígitos; se não houver dígitos suficientes, não apresenta
    if len(digitos) < 11:
        return 'NAO_APRESENTAR'

    return 'APRESENTAR'


df['status_dado'] = df['NU_CPF_PESSOA'].apply(classificar_status_cpf)

resumo_status = df['status_dado'].value_counts(dropna=False)
print('📊 Distribuição de status_dado:')
print(resumo_status)
print()
print(f"✅ Aptos (APRESENTAR): {(df['status_dado'] == 'APRESENTAR').sum()}")
print(f"🚫 Excluídos logicamente (NAO_APRESENTAR): {(df['status_dado'] == 'NAO_APRESENTAR').sum()}")

df[['NU_CPF_PESSOA', 'CPF_ANON', 'status_dado']].head(12)


In [ ]:
# 📊 Visualização didática do impacto do tratamento lógico
fig, ax = plt.subplots(figsize=(8, 5))
cores = {'APRESENTAR': '#2E86AB', 'NAO_APRESENTAR': '#E76F51'}
ordem = ['APRESENTAR', 'NAO_APRESENTAR']
contagem = df['status_dado'].value_counts().reindex(ordem).fillna(0)

bars = ax.bar(ordem, contagem.values, color=[cores[o] for o in ordem], edgecolor='black', alpha=0.85)
ax.set_title('🧹 Impacto do tratamento lógico (status_dado)', fontsize=14, fontweight='bold')
ax.set_ylabel('Quantidade de linhas')
ax.set_xlabel('Status')

for b in bars:
    ax.annotate(f'{int(b.get_height())}',
                xy=(b.get_x() + b.get_width() / 2, b.get_height()),
                ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


# 🧩 5. Registro Representativo Único (`REP_DADO_UNICO`)

## 🧠 Por que isso é necessário?
Quando uma ocorrência combina **múltiplos** `DS_GRUPO_ENQTO` com **múltiplos** `DS_TIPO_APREENSAO`, a base pode sofrer **produto cartesiano** (linhas se multiplicam).

Se somarmos a quantidade apreendida em todas as linhas, **inflamos o total**.

### ✅ Estratégia (sem apagar linhas)
1. Agrupar pela ocorrência/apreensão:
   - `NU_CPF_PESSOA`
   - `DS_DATA_CURTA`
   - `DS_TIPO_APREENSAO`
   - `QTDE APREENSâO OCORRÊNCIA`
2. Marcar **apenas a primeira linha** de cada grupo com `"U"`
3. Demais linhas do mesmo grupo ficam `None` (vazio)
4. Totais oficiais: filtrar `REP_DADO_UNICO == 'U'`

> **🎯 Resultado:** estrutura da tabela preservada + totais confiáveis.


In [ ]:
# 🧩 Função para marcar o registro representativo único

COL_QTDE = 'QTDE APREENSâO OCORRÊNCIA'

CHAVES_GRUPO = [
    'NU_CPF_PESSOA',
    'DS_DATA_CURTA',
    'DS_TIPO_APREENSAO',
    COL_QTDE,
]


def marcar_rep_dado_unico(dataframe, chaves=None):
    """Cria/atualiza a coluna REP_DADO_UNICO.

    Para cada grupo (ocorrência + características da apreensão),
    marca 'U' apenas na primeira linha; demais recebem None.
    """
    chaves = chaves or CHAVES_GRUPO
    out = dataframe.copy()

    # Garante que as chaves existam
    faltando = [c for c in chaves if c not in out.columns]
    if faltando:
        raise KeyError(f'Colunas de agrupamento ausentes: {faltando}')

    # Inicializa vazio
    out['REP_DADO_UNICO'] = None

    # groupby().cumcount() == 0 identifica a primeira linha de cada grupo
    primeira = out.groupby(chaves, dropna=False).cumcount() == 0
    out.loc[primeira, 'REP_DADO_UNICO'] = 'U'

    return out


df = marcar_rep_dado_unico(df)

n_u = (df['REP_DADO_UNICO'] == 'U').sum()
n_dup = df['REP_DADO_UNICO'].isna().sum()
print('✅ Coluna REP_DADO_UNICO criada!')
print(f'🏷️  Registros únicos (U): {n_u}')
print(f'📄 Linhas complementares (vazias): {n_dup}')
print(f'📐 Total de linhas: {len(df)}')

df[CHAVES_GRUPO + ['DS_GRUPO_ENQTO', 'REP_DADO_UNICO']].head(15)


In [ ]:
# 🔬 Demonstração: por que filtrar REP_DADO_UNICO == 'U' importa

# Base analítica oficial
mask_valida = (df['status_dado'] == 'APRESENTAR') & (df['REP_DADO_UNICO'] == 'U')
df_analise = df.loc[mask_valida].copy()

# Garante quantidade numérica
df_analise[COL_QTDE] = pd.to_numeric(df_analise[COL_QTDE], errors='coerce').fillna(0)

soma_com_cartesiano = pd.to_numeric(df.loc[df['status_dado'] == 'APRESENTAR', COL_QTDE], errors='coerce').fillna(0).sum()
soma_unica = df_analise[COL_QTDE].sum()

print('🧮 Comparativo de totais (apenas status APRESENTAR):')
print(f'  • Soma SEM filtro REP_DADO_UNICO: {soma_com_cartesiano:,.0f}')
print(f'  • Soma COM filtro REP_DADO_UNICO == "U": {soma_unica:,.0f}')
print(f'  • Diferença (possível inflação): {soma_com_cartesiano - soma_unica:,.0f}')
print()
print(f'📏 Linhas na base analítica (APRESENTAR ∩ U): {len(df_analise)}')


# 📈 6. Análises e Visualizações

A partir daqui usamos **`df_analise`**:
- ✅ `status_dado == 'APRESENTAR'`
- ✅ `REP_DADO_UNICO == 'U'`

### 🎯 Perguntas de negócio
1. **⏳ Temporal:** como as principais apreensões se distribuem no tempo?
2. **🗺️ Espacial:** quais UFs concentram as principais apreensões?
3. **🛣️ Rodovias:** qual rodovia é mais relevante em volume apreendido?


In [ ]:
# 📅 Preparação da dimensão temporal
# DS_DATA_CURTA pode vir como texto/data — forçamos datetime

df_analise['DATA'] = pd.to_datetime(df_analise['DS_DATA_CURTA'], errors='coerce', dayfirst=True)
df_analise['ANO_MES'] = df_analise['DATA'].dt.to_period('M').astype(str)
df_analise['ANO'] = df_analise['DATA'].dt.year
df_analise['MES'] = df_analise['DATA'].dt.month

print('📅 Cobertura temporal:')
print(f"  Mínimo: {df_analise['DATA'].min()}")
print(f"  Máximo: {df_analise['DATA'].max()}")
print(f"  Nulos de data: {df_analise['DATA'].isna().sum()}")

# UF de trabalho: prefere sigla; se vazia, usa descrição
df_analise['UF'] = df_analise['SG_UF_LOCAL_OCORRENCIA'].fillna(df_analise['DS_UF_LOCAL_OCORRENCIA'])
df_analise['UF'] = df_analise['UF'].astype(str).str.strip().str.upper().replace({'NAN': np.nan, 'NONE': np.nan})

print('\n🗺️ UFs encontradas (amostra):', df_analise['UF'].dropna().unique()[:15])


## ⏳ 6.1 Distribuição temporal das principais apreensões

> **Técnica:** identificar o Top N de `DS_TIPO_APREENSAO` por quantidade e plotar série temporal (`plt.plot` / Seaborn `lineplot`).


In [ ]:
# 🏆 Top tipos de apreensão por quantidade total
TOP_N = 5  # 👈 ajuste se quiser mais/menos categorias

ranking_tipos = (
    df_analise.groupby('DS_TIPO_APREENSAO', dropna=False)[COL_QTDE]
    .sum()
    .sort_values(ascending=False)
)

print('🏆 Ranking geral — quantidade apreendida por tipo:')
display(ranking_tipos.head(10).to_frame('QTDE_TOTAL'))

principais_tipos = ranking_tipos.head(TOP_N).index.tolist()
print('\n📌 Principais apreensões selecionadas para a série temporal:')
for i, t in enumerate(principais_tipos, 1):
    print(f'  {i}. {t}')


In [ ]:
# 📈 Série temporal (mês) das principais apreensões
filtro_tempo = df_analise['DS_TIPO_APREENSAO'].isin(principais_tipos) & df_analise['ANO_MES'].notna()
temporal = (
    df_analise.loc[filtro_tempo]
    .groupby(['ANO_MES', 'DS_TIPO_APREENSAO'], as_index=False)[COL_QTDE]
    .sum()
    .sort_values('ANO_MES')
)

plt.figure(figsize=(14, 6))
sns.lineplot(
    data=temporal,
    x='ANO_MES',
    y=COL_QTDE,
    hue='DS_TIPO_APREENSAO',
    marker='o',
    linewidth=2.2
)
plt.title('⏳ Distribuição temporal das principais apreensões', fontsize=14, fontweight='bold')
plt.xlabel('Ano-Mês')
plt.ylabel('Quantidade apreendida (registros U)')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Tipo de apreensão', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

# 📋 Tabela complementar (pivot)
pivot_temporal = temporal.pivot_table(
    index='ANO_MES',
    columns='DS_TIPO_APREENSAO',
    values=COL_QTDE,
    aggfunc='sum',
    fill_value=0
)
print('📋 Tabela (Ano-Mês × Tipo) — principais apreensões:')
display(pivot_temporal)


## 🗺️ 6.2 Distribuição espacial por UF das principais apreensões

> **Técnica:** `groupby` + `pivot_table` + `sns.heatmap` / barras empilhadas para ler concentração geográfica.


In [ ]:
# 🗺️ Concentração por UF × tipo (principais apreensões)
filtro_geo = df_analise['DS_TIPO_APREENSAO'].isin(principais_tipos) & df_analise['UF'].notna()

geo = (
    df_analise.loc[filtro_geo]
    .groupby(['UF', 'DS_TIPO_APREENSAO'], as_index=False)[COL_QTDE]
    .sum()
)

pivot_geo = geo.pivot_table(
    index='UF',
    columns='DS_TIPO_APREENSAO',
    values=COL_QTDE,
    aggfunc='sum',
    fill_value=0
)

# Ordena UFs pela soma total
pivot_geo['__TOTAL__'] = pivot_geo.sum(axis=1)
pivot_geo = pivot_geo.sort_values('__TOTAL__', ascending=False)
totais_uf = pivot_geo['__TOTAL__'].copy()
pivot_geo = pivot_geo.drop(columns='__TOTAL__')

print('🗺️ Ranking de UFs (soma das principais apreensões):')
display(totais_uf.head(10).to_frame('QTDE_TOTAL'))

# Heatmap
plt.figure(figsize=(12, max(5, 0.45 * len(pivot_geo))))
sns.heatmap(pivot_geo, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.4)
plt.title('🗺️ Distribuição espacial (UF × principais apreensões)', fontsize=14, fontweight='bold')
plt.xlabel('Tipo de apreensão')
plt.ylabel('UF')
plt.tight_layout()
plt.show()


In [ ]:
# 📊 Barras: Top UFs no total apreendido (principais tipos)
top_ufs = totais_uf.head(10)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=top_ufs.values, y=top_ufs.index, orient='h', color='#2E86AB', ax=ax)
ax.set_title('🏅 Top 10 UFs — volume das principais apreensões', fontsize=14, fontweight='bold')
ax.set_xlabel('Quantidade apreendida (registros U)')
ax.set_ylabel('UF')

for i, v in enumerate(top_ufs.values):
    ax.text(v, i, f' {v:,.0f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()


## 🛣️ 6.3 Qual rodovia é mais relevante?

> **Critério:** maior soma de `QTDE APREENSâO OCORRÊNCIA` nos registros únicos válidos (`REP_DADO_UNICO == 'U'`).

Complementamos com composição por tipo de apreensão nas rodovias líderes.


In [ ]:
# 🛣️ Ranking de rodovias por quantidade apreendida
rodovias = (
    df_analise.groupby('DS_RODOVIA', dropna=False)[COL_QTDE]
    .agg(qtde_total='sum', n_registros='count')
    .sort_values('qtde_total', ascending=False)
)

# Remove rótulos vazios óbvios
rodovias = rodovias[~rodovias.index.astype(str).str.strip().str.lower().isin(['', 'nan', 'none', 'nao informado', 'não informado'])]

print('🛣️ Ranking de rodovias (top 15):')
display(rodovias.head(15))

if len(rodovias) == 0:
    print('⚠️ Nenhuma rodovia válida encontrada para ranquear.')
else:
    campeã = rodovias.index[0]
    print(f"\n🏆 Rodovia mais relevante: {campeã}")
    print(f"   Quantidade total apreendida: {rodovias.iloc[0]['qtde_total']:,.0f}")
    print(f"   Nº de registros únicos (U): {int(rodovias.iloc[0]['n_registros'])}")


In [ ]:
# 📊 Visualização — Top rodovias
TOP_RODOVIAS = 10
top_rod = rodovias.head(TOP_RODOVIAS).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Barras horizontais
sns.barplot(
    data=top_rod,
    x='qtde_total',
    y='DS_RODOVIA',
    color='#52B788',
    ax=axes[0]
)
axes[0].set_title(f'🛣️ Top {TOP_RODOVIAS} rodovias por quantidade apreendida', fontweight='bold')
axes[0].set_xlabel('Quantidade apreendida')
axes[0].set_ylabel('Rodovia')

# Composição por tipo na rodovia campeã (se houver)
if len(rodovias) > 0:
    campeã = rodovias.index[0]
    comp = (
        df_analise.loc[df_analise['DS_RODOVIA'] == campeã]
        .groupby('DS_TIPO_APREENSAO')[COL_QTDE]
        .sum()
        .sort_values(ascending=False)
        .head(8)
    )
    axes[1].pie(
        comp.values,
        labels=comp.index,
        autopct='%1.1f%%',
        startangle=90,
        wedgeprops={'edgecolor': 'white'}
    )
    axes[1].set_title(f'🧩 Composição por tipo — {campeã}', fontweight='bold')
else:
    axes[1].axis('off')

plt.tight_layout()
plt.show()


## 🔎 6.4 Extras didáticos (funções da aula)

Aqui reforçamos técnicas vistas em 1.1 e 1.2:
- 📊 `groupby().agg()` com múltiplas métricas
- 🧮 estatísticas com **NumPy**
- 📦 `boxplot` / `hist` para distribuição das quantidades


In [ ]:
# 📊 Agregações múltiplas por tipo de apreensão
agg_tipos = (
    df_analise.groupby('DS_TIPO_APREENSAO')[COL_QTDE]
    .agg(total='sum', media='mean', mediana='median', registros='count', desvio='std')
    .sort_values('total', ascending=False)
)

print('📋 Agregações por DS_TIPO_APREENSAO:')
display(agg_tipos.head(10))

# 🧮 Estatísticas com NumPy sobre as quantidades
qtdes = df_analise[COL_QTDE].to_numpy(dtype=float)
print('\n🧮 Estatísticas NumPy (quantidades nos registros U):')
print(f'  Média: {np.mean(qtdes):.2f}')
print(f'  Mediana: {np.median(qtdes):.2f}')
print(f'  Desvio padrão: {np.std(qtdes):.2f}')
print(f'  Mínimo: {np.min(qtdes):.2f}')
print(f'  Máximo: {np.max(qtdes):.2f}')
print(f'  25º percentil: {np.percentile(qtdes, 25):.2f}')
print(f'  75º percentil: {np.percentile(qtdes, 75):.2f}')


In [ ]:
# 📦 Distribuição das quantidades (hist + boxplot)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Histograma
ax1.hist(df_analise[COL_QTDE], bins=20, color='#52B788', alpha=0.8, edgecolor='black')
ax1.set_title('📊 Histograma — QTDE apreendida (registros U)', fontweight='bold')
ax1.set_xlabel('Quantidade')
ax1.set_ylabel('Frequência')

# Boxplot dos principais tipos
dados_box = df_analise[df_analise['DS_TIPO_APREENSAO'].isin(principais_tipos)]
sns.boxplot(data=dados_box, x='DS_TIPO_APREENSAO', y=COL_QTDE, ax=ax2, color='#2E86AB')
ax2.set_title('📦 Boxplot — principais tipos', fontweight='bold')
ax2.set_xlabel('Tipo de apreensão')
ax2.set_ylabel('Quantidade')
ax2.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
# 🧾 Amostra final “pronta para relatório” (dados anonimizados)
colunas_relatorio = [
    'CPF_ANON', 'NOME_INICIAIS', 'DS_DATA_CURTA', 'UF', 'DS_RODOVIA',
    'DS_TIPO_APREENSAO', COL_QTDE, 'status_dado', 'REP_DADO_UNICO'
]
colunas_relatorio = [c for c in colunas_relatorio if c in df_analise.columns]

print('🔐 Amostra para relatório (sem CPF/nome em claro):')
display(df_analise[colunas_relatorio].head(15))

# 💾 Exportação opcional
# df_analise[colunas_relatorio].to_excel('apreensoes_analise_anonimizada.xlsx', index=False)
# print('✅ Arquivo exportado: apreensoes_analise_anonimizada.xlsx')


# 🎓 7. Conclusão

### ✅ O que este notebook entregou
- 🎭 Anonimização de **CPF** (`123.***.***-*5`) e **nome** (iniciais)
- 🧹 Tratamento lógico com **`status_dado`** (sem apagar linhas)
- 🧩 Marcação de **`REP_DADO_UNICO = 'U'`** para evitar inflação por produto cartesiano
- 📈 Análises:
  1. Distribuição **temporal** das principais apreensões
  2. Distribuição **espacial por UF**
  3. **Rodovia** mais relevante

### 🧠 Como interpretar os resultados
- Use sempre `df_analise` (ou o filtro `APRESENTAR` ∩ `U`) para totais
- Compare rankings (tipo, UF, rodovia) e valide com as tabelas pivot
- Se uma rodovia liderar com poucos registros, investigue outliers de quantidade

---

### 🚀 Próximos passos sugeridos
1. Cruzar com `DS_GRUPO_ENQTO` / `DS_COMPLEMENTO` para tipologias mais finas
2. Criar mapa coroplético por UF (bibliotecas geo)
3. Automatizar o pipeline em um script `.py` reutilizável

---

> **✨ Lembrete:** *“Dados bem tratados contam histórias melhores.”*


## 📋 Cheat Sheet deste desafio

### 🎭 Anonimização
```python
anonimizar_cpf('123.456.789-05')  # -> '123.***.***-*5'
iniciais_nome('Joao Jose da Silva Nogueira')  # -> 'JJSN'
```

### 🧹 Status lógico
```python
df['status_dado'] = df['NU_CPF_PESSOA'].apply(classificar_status_cpf)
# APRESENTAR | NAO_APRESENTAR
```

### 🧩 Registro único
```python
df = marcar_rep_dado_unico(df)
df_analise = df[(df['status_dado']=='APRESENTAR') & (df['REP_DADO_UNICO']=='U')]
```

### 📊 Agregação e gráfico
```python
df_analise.groupby('DS_TIPO_APREENSAO')[COL_QTDE].sum()
sns.lineplot(data=temporal, x='ANO_MES', y=COL_QTDE, hue='DS_TIPO_APREENSAO')
sns.heatmap(pivot_geo, annot=True, fmt='.0f', cmap='YlOrRd')
```

**💡 Mantenha este notebook como modelo para próximas bases operacionais!**
